In [ ]:
import pandas as pd

def ground_truths_escenario1(log_text):
    # Convertir a mayúsculas para evitar problemas de case sensitive
    log = str(log_text).upper()
    
    # Regla 1: Acceso normal de alumno (uid=1000)
    if 'TYPE=SYSCALL' in log and any(cmd in log for cmd in ['COMM="CAT"', 'COMM="TAIL"', 'COMM="GREP"']) and 'UID=1000' in log:
        return ['BAJO', 'INFO', 'INFORMACIÓN']
        
    # Regla 2: Ejecución de sudo por root (euid=0)
    elif 'TYPE=SYSCALL' in log and 'COMM="SUDO"' in log and 'EUID=0' in log:
        return ['MEDIO']
        
    # Regla 3: Eventos auxiliares que no tienen riesgo directo
    elif 'TYPE=CWD' in log or 'TYPE=PROCTITLE' in log:
        return ['INFO', 'INFORMACIÓN']
        
    # Por defecto, cualquier otro evento de soporte del sistema es ruido/info
    return ['INFORMACIÓN', 'INFO', 'BAJO']

def evaluar_por_orden(path_completo, path_evaluar, funcion_heuristica):
    """
    Evalúa la precisión comparando fila por fila basándose en el orden estricto de los CSV.
    """
    df_completo = pd.read_csv(path_completo)
    df_evaluar = pd.read_csv(path_evaluar)
    
    # Verificación de integridad
    if len(df_completo) != len(df_evaluar):
        print(f"¡ALERTA!: Archivos desfasados. Completo: {len(df_completo)} | A evaluar: {len(df_evaluar)}")
        return
        
    # Construir la lista de la verdad absoluta
    verdad_en_orden = df_completo['Log'].apply(funcion_heuristica).tolist()
    
    aciertos = 0
    total_validos = 0
    
    # Comparar fila por fila
    for i in range(len(df_evaluar)):
        prediccion_llm = str(df_evaluar.loc[i, 'Riesgo']).upper().strip()
        etiquetas_reales = verdad_en_orden[i]
        
        if 'ERROR' not in etiquetas_reales:
            total_validos += 1
            if prediccion_llm in etiquetas_reales:
                aciertos += 1
                
    precision = (aciertos / total_validos) * 100 if total_validos > 0 else 0
    print(f"Precisión para {path_evaluar}: {precision:.2f}% ({aciertos}/{total_validos})")
    return precision


In [ ]:
# --- EJECUCIÓN ---
# Definir la ruta del archivo que tiene la verdad (RAW Completo)
ruta_verdad = '../results/prompt1/escenario1_resultados_raw_completo_phi3mini.csv'

# Evaluar el RAW Completo contra sí mismo para la nota base
evaluar_por_orden(ruta_verdad, ruta_verdad, ground_truths_escenario1)

# Evaluar los otros formatos
evaluar_por_orden(ruta_verdad, '../results/prompt1/escenario1_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario1)
evaluar_por_orden(ruta_verdad, '../results/prompt1/escenario1_resultados_json_reducido_phi3mini.csv', ground_truths_escenario1)
evaluar_por_orden(ruta_verdad, '../results/prompt2/escenario1_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario1)
evaluar_por_orden(ruta_verdad, '../results/prompt3/escenario1_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario1)


Precisión para ../results/prompt1/escenario1_resultados_raw_completo_phi3mini.csv: 73.38% (102/139)
Precisión para ../results/prompt1/escenario1_resultados_raw_reducido_phi3mini.csv: 87.05% (121/139)
Precisión para ../results/prompt1/escenario1_resultados_json_reducido_phi3mini.csv: 66.91% (93/139)
Precisión para ../results/prompt2/escenario1_resultados_raw_reducido_phi3mini.csv: 74.10% (103/139)
Precisión para ../results/prompt3/escenario1_resultados_raw_reducido_phi3mini.csv: 92.81% (129/139)


92.80575539568345